# PRACTICAL 4: Request Labeling & Classification

## Objective
Categorize log requests in `cleaned_logs.csv` by applying rule-based regex signatures to detect threat vectors:
1. **SQL Injection (`sqli`)**: Signatures like `'`, `--`, `OR 1=1`, `UNION SELECT`, `DROP TABLE`.
2. **Path Traversal (`path_traversal`)**: Signatures like `../`, `..\`, `/etc/passwd`, `boot.ini`.
3. **Automated Scanner / Brute Force (`brute_force`)**: Known security scanning tools (`sqlmap`, `nikto`, `gobuster`, `dirbuster`, `nmap`, `curl`, `hydra`).
4. **Benign (`benign`)**: Normal user traffic.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

plt.style.use('seaborn-v0_8')
%matplotlib inline

### 1. Load Cleaned Dataset

In [ ]:
input_file = "cleaned_logs.csv"
print(f"Loading dataset from '{input_file}'...")
df = pd.read_csv(input_file, low_memory=False)
print(f"Total Rows Loaded to Label: {len(df):,}")

### 2. Apply Classification Signatures

In [ ]:
df['resource_requested'] = df['resource_requested'].fillna('').astype(str)
df['user_agent'] = df['user_agent'].fillna('').astype(str)
combined_text = df['resource_requested'] + " " + df['user_agent']

# Patterns
sqli_pattern = r"(?:'|\%27)|(?:\-\-) |(?:\bOR\b.*=.*)|(?:\bUNION\b.*SELECT\b)|(?:\bDROP\b.*TABLE\b)"
path_traversal_pattern = r"(?:\.\./|\.\.\\|/etc/passwd|boot\.ini)"
bot_scanner_pattern = r"(?:sqlmap|nikto|gobuster|dirbuster|nmap|python-urllib|curl|hydra)"

is_sqli = combined_text.str.contains(sqli_pattern, regex=True, case=False)
is_path_traversal = combined_text.str.contains(path_traversal_pattern, regex=True, case=False)
is_brute_force = df['user_agent'].str.contains(bot_scanner_pattern, regex=True, case=False)

df['label'] = 'benign'
df.loc[is_brute_force, 'label'] = 'brute_force'
df.loc[is_path_traversal, 'label'] = 'path_traversal'
df.loc[is_sqli, 'label'] = 'sqli'

### 3. Display Distribution & Export Labeled Data

In [ ]:
label_counts = df['label'].value_counts()
print("--- LABEL DISTRIBUTION --- ")
print(label_counts)

os.makedirs("../plots", exist_ok=True)
plt.figure(figsize=(10, 6))
sns.barplot(x=label_counts.index, y=label_counts.values, palette='magma', hue=label_counts.index, legend=False)
plt.title("Distribution of Security Classification Labels", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Assigned Label", fontsize=12)
plt.ylabel("Number of Logs", fontsize=12)
plt.yscale('log')
for i, val in enumerate(label_counts.values):
    plt.text(i, val * 1.1, f"{val:,}", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig("../plots/plot_p4_label_distribution.png", dpi=300)
plt.show()

output_file = "labeled_logs.csv"
df.to_csv(output_file, index=False)
print(f"Saved labeled dataset to '{output_file}'!")

### 4b. Bot / Tool Sub-Classification — *which* bot attacked?
(Shared bot_taxonomy module; bot_family is metadata, label stays the ML target.)

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from bot_taxonomy import classify_bot, family_label

df["bot_family"] = df["user_agent"].astype(str).map(classify_bot)
df.loc[(df["label"] == "brute_force") & df["bot_family"].isin(["browser", "unknown"]), "bot_family"] = "other_bot"

tool_summary = df["bot_family"].value_counts().reset_index()
tool_summary.columns = ["Attacking Tool", "Requests"]
tool_summary["Attacking Tool"] = tool_summary["Attacking Tool"].map(family_label)
tool_summary["Share %"] = (tool_summary["Requests"] / len(df) * 100).round(2)
tool_summary

### 4c. SQLi payload-variant sub-classification — *which* technique?
(Shared sqli_taxonomy module; sqli_variant is metadata, label stays the ML target.)

In [ ]:
from sqli_taxonomy import classify_sqli, variant_label

df["sqli_variant"] = "-"
is_sqli = df["label"] == "sqli"
df.loc[is_sqli, "sqli_variant"] = df.loc[is_sqli, "resource_requested"].astype(str).map(classify_sqli)

variant_summary = df[df["sqli_variant"] != "-"
]["sqli_variant"].value_counts().reset_index()
variant_summary.columns = ["Payload Variant", "Requests"]
variant_summary["Payload Variant"] = variant_summary["Payload Variant"].map(variant_label)
variant_summary["Share %"] = (variant_summary["Requests"] / len(df) * 100).round(2)
variant_summary